In [18]:
import numpy as np
import pandas as pd
import folium
from sklearn.neighbors import BallTree
import requests
from tqdm import tqdm
import time
import json
import random
from shapely.geometry import Point, Polygon
import geopandas as gpd

#### 데이터 가져오기

In [19]:
DATA_PATH = 'D:/workspaces/00_Project/crawlProject/test_project/project/data/'

# DATA_PATH = 'C:/crawling/test/project/data/'


In [20]:
def get_all_data(data_list):
    '''
    Put all data in dictionary
    '''
    dfs = {}

    for data_name in data_list:
        df = pd.read_csv(DATA_PATH + f"{data_name}/df_{data_name}.csv")
        dfs[data_name] = df
    return dfs

In [21]:
def get_df_grid(data_name):
    df_grid = pd.read_csv(DATA_PATH + f'grid/{data_name}.csv')
    return df_grid

In [22]:
def get_df_population(df_grid):
    
    # 1. 데이터 불러오기
    df_pop = pd.read_csv(DATA_PATH + 'population/df_population_raw.csv')
    
    gdf_grid = gpd.GeoDataFrame(df_grid, geometry=[Point(xy) for xy in zip(df_grid['center_lng'], df_grid['center_lat'])], crs="EPSG:4326")
    gdf_pop = gpd.GeoDataFrame(df_pop, geometry=[Point(xy) for xy in zip(df_pop['center_lng'], df_pop['center_lat'])], crs="EPSG:4326")

    # 3. grid 미터좌표계 변환
    gdf_grid = gdf_grid.to_crs(epsg=5179)
    gdf_pop = gdf_pop.to_crs(epsg=5179)
    
    # 4. 인구 밀집도 조인 
    gdf_pop_join = gpd.sjoin_nearest(gdf_grid, gdf_pop[['밀집도', 'geometry']], how='left')
    gdf_pop_join = gdf_pop_join.rename(columns={'밀집도': 'population_density'})
    if 'index_right' in gdf_pop_join.columns: gdf_pop_join.drop(columns='index_right', inplace=True)

    # 5. 결과는 다시 위경도(4326)로 복구하여 데이터프레임화
    gdf_pop_join = gdf_pop_join.to_crs(epsg=4326)
    
    # 6. 최종 출력용 DF 정리 및 csv export
    df_population = pd.DataFrame(gdf_pop_join.drop(columns='geometry'))
    df_population = df_population[['grid_id', 'center_lat', 'center_lng', 'population_density']]
    df_population = df_population.fillna(0)
    df_population.to_csv('data/df_population.csv', encoding = 'utf-8', index=False)

    return df_population

In [23]:
df_grid = get_df_grid('grid2')
df_population = get_df_population(df_grid)

In [24]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_population['population_density_normalized'] = scaler.fit_transform(df_population[['population_density']])
df_population

,grid_id,center_lat,center_lng,population_density,population_density_normalized
0,1,37.485385,127.017549,29571.48,0.138029
1,2,37.485385,127.018684,29571.48,0.138029
2,3,37.486286,127.016413,59912.72,0.280215
3,4,37.486286,127.017549,29571.48,0.138029
4,5,37.486286,127.018684,29571.48,0.138029
...,...,...,...,...,...
1910,1911,37.530431,127.032313,447.59,0.001547
1911,1912,37.531331,127.026634,803.86,0.003217
1912,1913,37.531331,127.027770,55388.75,0.259014
1913,1914,37.531331,127.028906,55388.75,0.259014


In [25]:
# 50m 격자 <-> 인구, 토지 격자 sjoin 실행

def join_areas(grid, population, area_grid):

    # 1. 데이터 불러오기
    df_50m = pd.read_csv(grid)    # 50m 격자 데이터  / geopandas 전환 필요
    df_pop = pd.read_csv(population)         # 인구밀집도 (격자 250m단위)  / geopandas 전환필요
    gdf_building_m = gpd.read_file(area_grid)     # 건물밀집도 (격자 500m단위)

    # 2. 데이터 geopandas 생성
    gdf_50m = gpd.GeoDataFrame(df_50m, geometry=[Point(xy) for xy in zip(df_50m['center_lng'], df_50m['center_lat'])], crs="EPSG:4326")
    gdf_pop_pt = gpd.GeoDataFrame(df_pop, geometry=[Point(xy) for xy in zip(df_pop['center_lng'], df_pop['center_lat'])], crs="EPSG:4326")

    # 3. grid 미터좌표계 변환
    gdf_50m_m = gdf_50m.to_crs(epsg=5179)
    gdf_pop_pt = gdf_pop_pt.to_crs(epsg=5179)

    # 4. 공간 조인 수행 (미터 단위에서 실행)
    # 건물 밀집도 조인
    gdf_final_m = gpd.sjoin(gdf_50m_m, gdf_building_m[['value', 'geometry']], how='left', predicate='within')
    gdf_final_m = gdf_final_m.rename(columns={'value': 'building_density'})
    if 'index_right' in gdf_final_m.columns: gdf_final_m.drop(columns='index_right', inplace=True)
    
    # 인구 밀집도 조인 (이제 경고가 뜨지 않습니다)
    gdf_final_m = gpd.sjoin_nearest(gdf_final_m, gdf_pop_pt[['밀집도', 'geometry']], how='left')
    gdf_final_m = gdf_final_m.rename(columns={'밀집도': 'population_density'})
    if 'index_right' in gdf_final_m.columns: gdf_final_m.drop(columns='index_right', inplace=True)
    
    # 5. 결과는 다시 위경도(4326)로 복구하여 데이터프레임화
    gdf_final_4326 = gdf_final_m.to_crs(epsg=4326)
    
    # 6. 최종 출력용 DF 정리 및 csv export
    final_df = pd.DataFrame(gdf_final_4326.drop(columns='geometry'))
    final_df = final_df[['grid_id', 'center_lat', 'center_lng', 'building_density', 'population_density']]
    final_df = final_df.fillna(0)
    final_df.to_csv('data/final_df.csv', encoding = 'utf-8', index=False)
    
    return final_df

#### 점수 산출

In [26]:
def building_cover(coords_grid, coords_building, RANGE_KM=1):
    """
    격자 좌표를 기준으로 반경 내 포함되는 건물 정보를 계산하는 함수

    Parameters
    ----------
    coords_grid : array-like (n, 2)
        격자 중심 좌표 (위도, 경도, degree)

    coords_building : array-like (m, 2)
        건물 좌표 (위도, 경도, degree)

    range_km : float, optional
        탐색 반경 (km 단위), 기본값 1km

    Returns
    -------
    df_result : pandas.DataFrame
        각 격자별 포함된 건물 개수와 인덱스 정보를 담은 데이터프레임
        - grid_id : 격자 인덱스
        - building_count : 포함된 건물 개수
        - building_indices : 포함된 건물 인덱스 리스트
    """

    # rad 변환
    grid_rad = np.deg2rad(coords_grid)
    building_rad = np.deg2rad(coords_building)

    # BallTree (건물 기준)
    tree = BallTree(building_rad, metric='haversine')

    # 반경 (km → rad)
    radius = RANGE_KM / 6371

    # 각 grid 기준 건물 찾기
    indices = tree.query_radius(grid_rad, r=radius)

    # 결과 정리
    building_indices = indices
    building_count = [len(idx) for idx in building_indices]

    df_result = pd.DataFrame({
        'grid_id': range(len(coords_grid)),
        'building_count': building_count,
        'building_indices': building_indices,
    })

    return df_result

In [27]:
def calc_score(dfs, df_grid, RANGE_KM):
    '''
    격자 점수 산출
    dfs : 데이터 딕셔너리
    RANGE_KM : 레이더 사정거리
    '''

    grid_lat_min = df_grid["sw_lat"].min()
    grid_lat_max = df_grid["ne_lat"].max()
    grid_lon_min = df_grid["sw_lng"].min()
    grid_lon_max = df_grid["ne_lng"].max()
   
    # 격자 내부에 있는 데이터만 필터링
    filtered_list = []

    for key, df in dfs.items():
        filtered = df[
            (df['latitude'] >= grid_lat_min) &
            (df['latitude'] <= grid_lat_max) &
            (df['longitude'] >= grid_lon_min) &
            (df['longitude'] <= grid_lon_max)
        ].copy()

        filtered_list.append(filtered)

    df_building = pd.concat(filtered_list, ignore_index=True)

    coords_building = df_building[['latitude', 'longitude']].values
    coords_grid = df_grid[['center_lat', 'center_lng']].values


    # 점수 산정
    df_result = building_cover(coords_grid, coords_building, RANGE_KM)

    scores = []
    for cover in df_result['building_indices'].values:
        sc = 0
        for grid_no in cover:
            sc += df_building.iloc[grid_no]['score']
        scores.append(sc)

    df_result['score'] = scores
    return df_result

In [28]:
def grid_cover_single(center_coord, all_coords, RANGE_KM=3):
    """
    center_coord: [lat, lng] - 기준이 되는 단일 좌표
    all_coords: (n,2) array - 전체 격자 좌표들
    range_km: 커버 반경 (km)
    """

    # degree → radian
    all_coords_rad = np.deg2rad(all_coords)
    center_rad = np.deg2rad(center_coord).reshape(1, -1)

    # BallTree 생성
    tree = BallTree(all_coords_rad, metric='haversine')

    # 반경 내 이웃 탐색
    radius = RANGE_KM / 6371
    indices = tree.query_radius(center_rad, r=radius)[0]

    # 자기 자신 제외 (center_coord와 동일한 좌표)
    indices = indices[~np.all(all_coords[indices] == center_coord, axis=1)]

    return indices  # 주변 격자의 인덱스 배열

In [30]:
##### TEST #######

def visualize(df_grid, dfs, rank_dic, RANGE_KM, ICON_MAP, show_rank=None, polygon_coords=None):

    # 격자 중심점 계산
    mid_lat = (df_grid["ne_lat"].max() + df_grid["sw_lat"].min()) / 2
    mid_lng = (df_grid["sw_lng"].min() + df_grid["ne_lng"].max()) / 2

    m = folium.Map(location=[mid_lat, mid_lng], zoom_start=14)

    # ── 구역 표시 ───────────────────────────────────────────────────
    if polygon_coords is not None:
        # 실제 다각형으로 표시
        folium.Polygon(
            locations=polygon_coords,   # [(lat, lng), ...]
            color="blue",
            weight=2,
            fill=True,
            fill_opacity=0.05,
            tooltip="격자 전체 영역"
        ).add_to(m)
        poly = Polygon([(lng, lat) for lat, lng in polygon_coords])
    else:
        # 기존 bounding box 방식
        grid_lat_min = df_grid["sw_lat"].min()
        grid_lat_max = df_grid["ne_lat"].max()
        grid_lon_min = df_grid["sw_lng"].min()
        grid_lon_max = df_grid["ne_lng"].max()
        folium.Rectangle(
            bounds=[[grid_lat_min, grid_lon_min], [grid_lat_max, grid_lon_max]],
            color="blue", weight=2, fill=True, fill_opacity=0.05,
            tooltip="격자 전체 영역"
        ).add_to(m)
        poly = None

    # ── 건물 마커 ───────────────────────────────────────────────────
    for key, df in dfs.items():
        if dfs[key]['score'][0] == 0:
            continue
        
        layer = folium.FeatureGroup(name=key)

        # 1차: bounding box로 빠르게 후보 추림
        if polygon_coords is not None:
            lat_list = [c[0] for c in polygon_coords]
            lng_list = [c[1] for c in polygon_coords]
            filtered = df[
                (df['latitude']  >= min(lat_list)) &
                (df['latitude']  <= max(lat_list)) &
                (df['longitude'] >= min(lng_list)) &
                (df['longitude'] <= max(lng_list))
            ].copy()
            # 2차: 실제 다각형 내부만
            mask = filtered.apply(
                lambda r: poly.contains(Point(r['longitude'], r['latitude'])), axis=1
            )
            filtered = filtered[mask]
        else:
            filtered = df[
                (df['latitude']  >= grid_lat_min) &
                (df['latitude']  <= grid_lat_max) &
                (df['longitude'] >= grid_lon_min) &
                (df['longitude'] <= grid_lon_max)
            ].copy()

        for _, row in filtered.iterrows():
            folium.Marker(
                location=[row['latitude'], row['longitude']],
                tooltip=row['name'],
                popup=folium.Popup(row['name'], max_width=200),
                icon=ICON_MAP.get(key, folium.Icon(color="gray", icon="question", prefix="fa"))
            ).add_to(layer)

        layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    # ── 레이더 위치 표시 ────────────────────────────────────────────
    def random_color():
        r, g, b = random.randint(50,200), random.randint(50,200), random.randint(50,200)
        return f"#{r:02x}{g:02x}{b:02x}"

    rank_items = list(rank_dic.items())
    if show_rank is not None:
        rank_items = rank_items[:show_rank]

    for rank, (key, item) in enumerate(rank_items, start=1):
        loca = df_grid.loc[key, ['center_lat', 'center_lng']].values
        color = random_color()

        folium.Marker(
            location=loca,
            tooltip=f"{rank}순위 레이더, 점수: {item}",
            popup=folium.Popup(f"{rank}순위 레이더 (grid_id: {key}, score: {item})", max_width=200),
            icon=folium.DivIcon(
                html=f"""
                    <div style="
                        background-color: {color}; color: white;
                        font-size: 13px; font-weight: bold;
                        width: 28px; height: 28px; border-radius: 50%;
                        display: flex; align-items: center; justify-content: center;
                        border: 2px solid white; box-shadow: 2px 2px 4px rgba(0,0,0,0.4);
                    ">{rank}</div>
                """,
                icon_size=(28, 28), icon_anchor=(14, 14)
            )
        ).add_to(m)

        folium.Circle(
            location=loca, radius=RANGE_KM * 1000,
            color=color, fill=True, fill_color=color, fill_opacity=0.15,
            tooltip=f"{rank}순위 커버 범위"
        ).add_to(m)

    m.save("map.html")
    print("저장 완료: map.html")

In [31]:
def calc_score(df_building_filtered, df_grid, RANGE_KM):
    '''
    격자 점수 산출
    dfs : 데이터 딕셔너리
    RANGE_KM : 레이더 사정거리
    '''

    coords_building = df_building_filtered[['latitude', 'longitude']].values
    coords_grid = df_grid[['center_lat', 'center_lng']].values

        
    # 점수 산정
    df_result = building_cover(coords_grid, coords_building, RANGE_KM)

    scores = []
    for cover in df_result['building_indices'].values:
        sc = 0
        for grid_no in cover:
            sc += df_building_filtered.iloc[grid_no]['score']
        scores.append(sc)

    df_result['score'] = scores
    return df_result

In [ ]:
def calc_rank(dfs, df_grid, RANGE_KM, radar_num=50, polygon_coords=None):
    """
    최적 레이더 설치 위치를 순위별로 계산하는 함수
    
    Args:
        dfs            : 건물 데이터프레임 딕셔너리 {key: DataFrame}
        df_grid        : 격자 정보 데이터프레임
        RANGE_KM       : 레이더 커버 반경 (km)
        radar_num      : 선정할 최대 레이더 수 (기본값 50)
        polygon_coords : 격자를 만들 때 사용한 꼭짓점 리스트
                         [(lat1, lng1), (lat2, lng2), ...] 형태
                         None 이면 기존 bounding box 방식으로 동작
    
    Returns:
        rank_dic      : {격자 인덱스: 점수} 순위별 레이더 위치
        max_radar_num : 실제 사용된 레이더 수
    """

    rank_dic = {}

    # ── 1. 격자 범위 계산 ──────────────────────────────────────────
    grid_lat_min = df_grid["sw_lat"].min()
    grid_lat_max = df_grid["ne_lat"].max()
    grid_lon_min = df_grid["sw_lng"].min()
    grid_lon_max = df_grid["ne_lng"].max()

    # ── 2. 격자 범위 내 건물만 필터링 ─────────────────────────────
    if polygon_coords is not None:
        poly = Polygon([(lng, lat) for lat, lng in polygon_coords])
        lat_list = [c[0] for c in polygon_coords]
        lng_list = [c[1] for c in polygon_coords]
        bb_lat_min, bb_lat_max = min(lat_list), max(lat_list)
        bb_lng_min, bb_lng_max = min(lng_list), max(lng_list)
    else:
        poly = None

    filtered_list = []

    for key, df in dfs.items():
        # 1차: bounding box로 빠르게 후보 추림
        if polygon_coords is not None:
            filtered = df[
                (df['latitude']  >= bb_lat_min) &
                (df['latitude']  <= bb_lat_max) &
                (df['longitude'] >= bb_lng_min) &
                (df['longitude'] <= bb_lng_max)
            ].copy()
            # 2차: 실제 다각형 내부만 남김
            mask = filtered.apply(
                lambda r: poly.contains(Point(r['longitude'], r['latitude'])), axis=1
            )
            filtered = filtered[mask]
        else:
            # 기존 bounding box 방식
            filtered = df[
                (df['latitude']  >= grid_lat_min) &
                (df['latitude']  <= grid_lat_max) &
                (df['longitude'] >= grid_lon_min) &
                (df['longitude'] <= grid_lon_max)
            ].copy()

        filtered_list.append(filtered)

    df_building_filtered = pd.concat(filtered_list, ignore_index=True)

    # 가중치가 0인 중요건물은 계산에서 제외
    df_building_filtered = df_building_filtered[df_building_filtered['score'] > 0]

    
    # 건물 소거용 임시 복사본 (반복마다 커버된 건물을 제거해나감)
    df_building_temp = df_building_filtered.copy()

    # ── 3. 순위별 최적 위치 선정 ───────────────────────────────────
    max_radar_num = radar_num  # 조기 종료 없을 경우 대비 기본값

    for i in range(radar_num):

        # 현재 남은 건물 기준으로 각 격자의 점수 계산
        df_result = calc_score(df_building_temp, df_grid, RANGE_KM)

        # 최고 점수 및 해당 격자 인덱스 추출
        max_score = df_result['score'].max()
        best_points = list(df_result[df_result['score'] == max_score].index)

        # ── 3-1. 동점 격자가 여러 개인 경우 → 커버 범위가 가장 넓은 격자 선택
        if len(best_points) != 1:
            cover_list = []
            for point in best_points:
                center_coord = df_grid.loc[point, ['center_lat', 'center_lng']].values
                all_coords   = df_grid[['center_lat', 'center_lng']].values
                pos_indices  = grid_cover_single(center_coord, all_coords, RANGE_KM)
                label_indices = df_grid.index[pos_indices]
                cover_list.append(label_indices)

            best_idx      = max(range(len(cover_list)), key=lambda x: len(cover_list[x]))
            position_grid = best_points[best_idx]

        # ── 3-2. 최고 점수 격자가 하나인 경우 → 바로 선택
        else:
            position_grid = best_points[0]

        # 선택된 격자 저장
        rank_dic[position_grid] = max_score
        pos = df_grid.loc[position_grid, ['center_lat', 'center_lng']].values

        # ── 4. 선택된 위치 기준으로 커버된 건물 제거 ──────────────
        all_building_coords  = df_building_temp[['latitude', 'longitude']].values
        building_pos_indices = grid_cover_single(pos, all_building_coords, RANGE_KM)

        covered_set = set(map(tuple, all_building_coords[building_pos_indices]))
        df_building_temp = df_building_temp[
            ~df_building_temp.apply(
                lambda r: (r['latitude'], r['longitude']) in covered_set, axis=1
            )
        ]

        # ── 5. 진행 상황 출력 ──────────────────────────────────────
        print('-' * 30)
        print(f'{i+1}순위')
        print(f'위치 : {pos}')
        print(f'점수 : {rank_dic[position_grid]}')
        print(f'남은 시설물 : {len(df_building_temp)}개')

        # 모든 건물이 커버되면 조기 종료
        if len(df_building_temp) == 0:
            max_radar_num = i + 1
            print('-' * 30)
            print(f'최대 radar 개수: {max_radar_num}')
            break

    return rank_dic, max_radar_num

In [33]:
def set_score(dfs, weight_dic):
    for tag in dfs.keys():
        dfs[tag]['score'] = [weight_dic[tag]] * len(dfs[tag]) 

In [34]:
def get_grid_bd_points(grid_name):
    with open(f'data/grid/{grid_name}_polygon.json') as f:
        data = json.load(f)
    return data['polygon_coords']

In [ ]:
# Get Data
data_list = ['bridge',
             'broadcast',
             'electricity',
             'factory',
             'hospital',
             'infra',
             'prison',
             'public',
             'science',
             'telecommunication',
             'transportation',
             'water',
             'school',
             'tunnel']
dfs = get_all_data(data_list)
grid= 'grid2'
df_grid = get_df_grid(grid)
grid_bd_points = get_grid_bd_points(grid)

# Set score
weight_dic = {'bridge' : 0.00187,
             'broadcast': 0.41787,
             'electricity':0.02198,
             'factory':0.00376,
             'hospital':0.00064,
             'infra':0.00075,
             'prison':0.02757,
             'public':0.01237,
             'science':0.08148,
             'telecommunication':0.14754,
             'transportation':0.00187,
             'water':0.17237,
             'school':0,
             'tunnel':0.0003}
set_score(dfs, weight_dic)

# Parameters
RANGE_KM = 2.0

# Rank calculation
rank_dic, max_radar_num = calc_rank(dfs, df_grid, RANGE_KM, radar_num=10, polygon_coords=grid_bd_points)

# Icon config
ICON_MAP = {
    "bridge":          folium.Icon(color="purple",   icon="road",              prefix="fa"),
    "broadcast":       folium.Icon(color="orange",   icon="broadcast-tower",   prefix="fa"),
    "electricity":     folium.Icon(color="green",    icon="bolt",              prefix="fa"),
    "factory":         folium.Icon(color="blue",     icon="industry",          prefix="fa"),
    "hospital":        folium.Icon(color="red",      icon="hospital",          prefix="fa"),
    "infra":           folium.Icon(color="darkblue", icon="cogs",              prefix="fa"),
    "prison":          folium.Icon(color="black",    icon="university",        prefix="fa"),
    "public":          folium.Icon(color="cadetblue",icon="building",          prefix="fa"),
    "science":         folium.Icon(color="pink",     icon="flask",             prefix="fa"),
    "telecommunication": folium.Icon(color="beige",  icon="satellite-dish",    prefix="fa"),
    "transportation":  folium.Icon(color="darkgreen",icon="train",             prefix="fa"),
    "water":           folium.Icon(color="lightblue",icon="tint",              prefix="fa"),
    "school":          folium.Icon(color="darkgreen",    icon="graduation-cap",    prefix="fa"),  # ✅ 추가
    "tunnel":          folium.Icon(color="gray",     icon="circle-o",          prefix="fa"),  # ✅ 추가
}

# Visualize
visualize(df_grid, dfs, rank_dic, RANGE_KM, ICON_MAP, show_rank=None, polygon_coords=grid_bd_points)
# visualize(df_grid, dfs, rank_dic, ICON_MAP, show_rank=max_radar_num, polygon_coords=grid_bd_points)

       name   latitude   longitude     tag    score
0     신사고가교  37.521797  127.017769      교량  0.00187
1   주흥교고가차도  37.509545  127.016079      교량  0.00187
2       삼성교  37.510340  127.068934      교량  0.00187
3      서초3교  37.487382  127.024564      교량  0.00187
4       서초교  37.501466  127.019211      교량  0.00187
..      ...        ...         ...     ...      ...
67   호산여성병원  37.528568  127.033354      병원  0.00064
68      미병원  37.523464  127.031208      병원  0.00064
69    토마스병원  37.528513  127.035234      병원  0.00064
70     강남구청  37.517507  127.047375  지방행정기관  0.01237
71    금융결재원  37.500442  127.038184      금융  0.14754

[72 rows x 5 columns]
------------------------------
1순위
위치 : [ 37.50971  127.042534]
점수 : 0.3468900000000001
남은 시설물 : 21개
------------------------------
2순위
위치 : [ 37.504304 127.022092]
점수 : 0.023150000000000004
남은 시설물 : 7개
------------------------------
3순위
위치 : [ 37.515115 127.047077]
점수 : 0.00507
남은 시설물 : 1개
------------------------------
4순위
위치 : [ 37.506106 127.04707

## APPENDIX
#### BallTree 방법

In [36]:
def make_cover_df(coords, range_km=3, include_self=False):
    """
    coords: (n,2) array [lat, lng] (degree)
    range_km: 커버 반경 (km)
    include_self: 자기 자신 포함 여부
    """

    if len(coords) == 0:
        return pd.DataFrame()

    # degree → rad
    coords_rad = np.deg2rad(coords)

    # BallTree 생성
    tree = BallTree(coords_rad, metric='haversine')

    # km → rad
    radius = range_km / 6371

    # 이웃 찾기
    indices = tree.query_radius(coords_rad, r=radius)

    # 자기 자신 처리
    if include_self:
        cover_indices = indices
    else:
        cover_indices = [idx[idx != i] for i, idx in enumerate(indices)]

    # 커버 개수
    cover_count = [len(idx) for idx in cover_indices]

    # 결과
    df_coverage = pd.DataFrame({
        'jammer_id': range(len(coords)),
        'cover_count': cover_count,
        'covered_points': cover_indices
    })

    return df_coverage